# 9. PMD cell by site

Part of the **[Fig. 2 chapter](fig2.md)** — see it for the panel-by-panel map. The first code cell sets `ENTEX_ROOT` and activates the no-overwrite guard (see the [Reproduction guide](../reproduce.md)). *Outputs shown are the author's original run.*


## 📥 Required input files

This notebook reads the following files (paths resolve from `ENTEX_ROOT`/`REF_ROOT`; the setup cell sets them). See the chapter's `inputs.md` for RAW-vs-derived tags.

- `f'{REF_ROOT}/hg38/fasta/hg38.altseq.bed'`  ·  _reference_
- `f'{indir}clustering/merged/L2_hiconly/Epi-TPB/100k3C_embed.h5ad'`  ·  _embedding h5ad_
- `f'{indir}clustering/merged/L2_hiconly/Epi-TPB/mergehic_rocpr.npy'`  ·  _other_


In [1]:
# === Reproduction setup — edit ENTEX_ROOT / REF_ROOT for your machine ===
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT)
os.chdir(f'{ENTEX_ROOT}/analysis')   # original working directory
import repro_guard                    # no-overwrite guard (default: skip existing)

[repro_guard] active — READ-ONLY (all writes skipped; inline figures still render)


In [2]:
import numpy as np
import pandas as pd
from glob import glob
from scipy.sparse import csr_matrix
from concurrent.futures import ProcessPoolExecutor, as_completed

import pysam
import anndata

import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'

import warnings
warnings.filterwarnings("ignore")


In [3]:
indir = f'{ENTEX_ROOT}/'


In [4]:
chrom, start, end = 'chr1', 1800000, 3800000


In [5]:
bed = pd.read_csv(f'{REF_ROOT}/hg38/fasta/hg38.altseq.bed', sep='\t', header=None, index_col=None)
bed = bed.loc[(bed[0]==chrom) & (bed[1]>=start) & (bed[2]<=end), [1,2]].values
bed = bed - start

In [6]:
adata = anndata.read_h5ad(f'{indir}clustering/merged/L2_hiconly/Epi-TPB/100k3C_embed.h5ad')
adata


AnnData object with n_obs × n_vars = 5228 × 0
    obs: 'FinalmCReads', 'mCHFrac', 'mCGFrac', 'CisLongContact', 'Cis/Trans', 'Donor', 'Tissue', 'celltype', 'ClusterTissue', 'tsne_0', 'tsne_1', 'leiden_cons', 'leiden_cv', 'leiden_init'
    obsm: '100k3C_pc8_seuratL2', '100k3C_pc8_seuratL2_tsne', '100k3C_pca', 'X_pca', 'X_tsne'

In [7]:
label_reorder = np.load(f'{indir}clustering/merged/L2_hiconly/Epi-TPB/mergehic_rocpr.npy', allow_pickle=True)
adata.obs['L2hic'] = label_reorder.copy()


In [8]:
adata.obs['allcpath'] = f'{ENTEX_ROOT}/allc/' + adata.obs.index + '.CGN-Both.allc.tsv.gz'


In [9]:
selc = (adata.obs['L2hic']=='c0')
print(selc.sum())

485


In [10]:
selc = np.ones(adata.shape[0]).astype(bool)
print(selc.sum())


5228


In [11]:
def load_allc(allc_path):
    global chrom, start, end
    idy, data = [], []
    with pysam.TabixFile(allc_path) as allc:
        allc_lines = allc.fetch(chrom, start, end)
        for line in allc_lines:
            _, pos, _, context, mc, cov, *_ = line.split("\t")
            pos, mc, cov = int(pos), int(mc), int(cov)
            if cov>1:
                continue
            idy.append(pos-start)
            data.append(mc)
    return idy,data


In [12]:
from concurrent.futures import ProcessPoolExecutor, as_completed

cpu = 32
with ProcessPoolExecutor(cpu) as executor:
    futures = {}
    for i, allc_path in enumerate(adata.obs.loc[selc, 'allcpath'].values):
        future = executor.submit(
            load_allc,
            allc_path=allc_path,
        )
        futures[future] = i

    idx, idy, data = [], [], []
    for future in as_completed(futures):
        i = futures[future]
        yy, zz = future.result()
        idx.append(np.ones(len(yy))*i)
        idy.append(yy)
        data.append(zz)
        # print(f'cell {i} finished')
        

[W::hts_idx_load3] [W::hts_idx_load3] The index file is older than the data file: /large_storage/zhoulab/zhoujt/project/ENTEx/allc/P_9216_AR_Plate5-1-D6-F14.CGN-Both.allc.tsv.gz.tbi
The index file is older than the data file: /large_storage/zhoulab/zhoujt/project/ENTEx/allc/P_9216_AR_Plate5-1-D6-F13.CGN-Both.allc.tsv.gz.tbi


[W::hts_idx_load3] The index file is older than the data file: /large_storage/zhoulab/zhoujt/project/ENTEx/allc/P_9213_AR_Plate7-1-K9-H14.CGN-Both.allc.tsv.gz.tbi


In [13]:
idx = np.concatenate(idx)
idy = np.concatenate(idy)
data = np.concatenate(data)
data = ((data - 0.5) * 2).astype(int)
data = csr_matrix((data, (idx,idy)), shape=[selc.sum(), end-start])
pos = np.where(data.getnnz(axis=0)>0)[0]
data = data[:, pos]


In [14]:
posfilter = np.ones(len(pos)).astype(bool)
for xx,yy in bed:
    posfilter[np.logical_and(pos>=xx, pos<=yy)] = False

print(np.sum(posfilter)/len(posfilter))

0.9829603230233086


In [15]:
data = data[:, posfilter]
pos = pos[posfilter]
print(data.shape)

(5228, 99567)


In [16]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.imshow(data.toarray(), cmap='bwr', aspect='auto', interpolation='none', vmin=-1, vmax=1)


In [17]:
fig, ax = plt.subplots()
sns.histplot(data.getnnz(axis=0), bins=150)

<Axes: ylabel='Count'>

In [18]:
# fig, ax = plt.subplots(figsize=(15,1), dpi=300)
# tmp = (data==1).sum(axis=0).A1
# tmp = tmp / (tmp + (data==-1).sum(axis=0).A1)
# tmp[data.getnnz(axis=0)>500] = 0
# bs = 25
# nbins = tmp.shape[0] // bs
# tmp = tmp[:(nbins*bs)].reshape((nbins, bs)).mean(axis=1)
# x = np.arange(tmp.shape[0])
# ax.plot(x, tmp, linewidth=1)
# ax.set_xlim([0, nbins])
# ax.fill_between(x, tmp, 0, where=tmp >= 0, facecolor='C0', interpolate=True)


In [19]:
def compute_simi(data):
    nbins = data.shape[1]
    tmp = (data==1).astype(int)
    inter_pos = (tmp.T).dot(tmp)
    tmp = (data==-1).astype(int)
    inter_neg = (tmp.T).dot(tmp)
    inter_cov = np.abs(data).T.dot(np.abs(data))
    simi = (inter_pos + inter_neg) / inter_cov
    simi[(np.arange(nbins), np.arange(nbins))] = 0
    return (np.nansum(simi)) / (nbins * nbins - np.isnan(simi).sum() - nbins)

def expect_simi(data):
    nbins = data.shape[1]
    tmp = (data==1).sum(axis=0).A1 / data.getnnz(axis=0)
    simi = tmp[:, None].dot(tmp[None, :]) + (1 - tmp[:, None]).dot(1 - tmp[None, :])
    simi[(np.arange(nbins), np.arange(nbins))] = 0
    return (np.nansum(simi)) / (nbins * nbins - np.isnan(simi).sum() - nbins)


In [20]:
cpu = 32
with ProcessPoolExecutor(cpu) as executor:
    futures = {}
    for i,xx in enumerate(np.arange(0, data.shape[1]-50, 25)):
        future = executor.submit(
            compute_simi,
            data=data[:, xx:(xx+50)],
        )
        futures[future] = i

    simi = np.zeros(len(futures))
    for future in as_completed(futures):
        i = futures[future]
        simi[i] = future.result()
        # print(f'chunk {i} finished')
        

In [21]:
cpu = 32
with ProcessPoolExecutor(cpu) as executor:
    futures = {}
    for i,xx in enumerate(np.arange(0, data.shape[1]-50, 25)):
        future = executor.submit(
            expect_simi,
            data=data[:, xx:(xx+50)],
        )
        futures[future] = i

    simi_exp = np.zeros(len(futures))
    for future in as_completed(futures):
        i = futures[future]
        simi_exp[i] = future.result()
        # print(f'chunk {i} finished')
        

In [22]:
fig, axes = plt.subplots(2, 1, figsize=(15,2), dpi=300)

ax = axes[0]
nbins = simi.shape[0]
x = np.arange(nbins)
ax.plot(x, simi_exp, linewidth=0.5, color='C1', alpha=0.8)
ax.plot(x, simi, linewidth=0.5, color='C0', alpha=0.8)
ax.set_xlim([0, nbins])
# ax.fill_between(x, tmp, 0, where=tmp >= 0, facecolor='C0', interpolate=True)

ax = axes[1]
tmp = (data==1).sum(axis=0).A1
tmp = tmp / data.getnnz(axis=0)
tmp[data.getnnz(axis=0)>500] = 0
bs = 25
nbins = tmp.shape[0] // bs
exp = (tmp)
tmp = tmp[:(nbins*bs)].reshape((nbins, bs)).mean(axis=1)
x = np.arange(tmp.shape[0])
ax.plot(x, tmp, linewidth=1)
ax.set_xlim([0, nbins])
ax.fill_between(x, tmp, 0, where=tmp >= 0, facecolor='C0', interpolate=True)


In [23]:
fig, axes = plt.subplots(2, 1, figsize=(15,2), dpi=300, sharex='all')

ax = axes[0]
nbins = simi.shape[0]
x = pos[:(nbins*bs)].reshape((nbins, bs)).mean(axis=1)
ax.plot(x, simi_exp, linewidth=0.5, color='C1', alpha=0.8)
ax.plot(x, simi, linewidth=0.5, color='C0', alpha=0.8)
ax.set_xlim([0, 2e6])
# ax.fill_between(x, tmp, 0, where=tmp >= 0, facecolor='C0', interpolate=True)

ax = axes[1]
tmp = (data==1).sum(axis=0).A1
tmp = tmp / data.getnnz(axis=0)
tmp[data.getnnz(axis=0)>500] = 0
tmp = tmp[:(nbins*bs)].reshape((nbins, bs)).mean(axis=1)
ax.plot(x, tmp, linewidth=1)
ax.set_xlim([0, 2e6])
ax.set_xticks(np.arange(0, 2e6+1, 2e5))
ax.set_xticklabels([f'{xx//100000}M' for xx in np.arange(start, end+1, 2e5)])
ax.fill_between(x, tmp, 0, where=tmp >= 0, facecolor='C0', interpolate=True)


In [24]:
data_pos = (data==1).astype(int)
data_neg = (data==-1).astype(int)
data_cov = np.abs(data)
ratio = data_pos.sum(axis=0).A1 / data_cov.sum(axis=0).A1

interpos = data_pos[:, :-1].multiply(data_pos[:, 1:]).sum(axis=0).A1
interneg = data_neg[:, :-1].multiply(data_neg[:, 1:]).sum(axis=0).A1
intercov = data_cov[:, :-1].multiply(data_cov[:, 1:]).sum(axis=0).A1


In [25]:
simi = (interpos + interneg) / intercov
simi_exp = ratio[:-1]*ratio[1:] + (1-ratio[:-1])*(1-ratio[1:])


In [26]:
# def compute_simi(i):
#     result = (data_pos[:,i].T.dot(data_pos[:,i+1]) + data_neg[:,i].T.dot(data_neg[:,i+1])) / data_cov[:,i].T.dot(data_cov[:,i+1])
#     return result.A1[0]

# def expect_simi(i):
#     return ratio[i]*ratio[i+1] + (1-ratio[i])*(1-ratio[i+1])


In [27]:
def bintrack(data, pos, start, end, bs):
    nnz = pos.shape[0]
    nbins = (end-start) // bs
    data = csr_matrix((data, (np.zeros(nnz), pos)), [1, end-start])
    data = data[:(bs*nbins)].reshape((-1, bs))
    data = data.sum(axis=1).A1 / data.getnnz(axis=1)
    data[np.isnan(data)] = 0
    return data


In [28]:
simi[np.isnan(simi)] = 0.5
simi_exp[np.isnan(simi_exp)] = 0.5


In [29]:
from scipy.stats import pearsonr
pearsonr(simi, simi_exp)

PearsonRResult(statistic=0.5530817234357834, pvalue=0.0)

In [30]:
fig, ax = plt.subplots(figsize=(4,4), dpi=300)
ax.scatter(simi_exp, simi, s=1, edgecolor='none', rasterized=True)
ax.plot([0.5, 1], [0.5, 1], 'k')

In [31]:
fig, axes = plt.subplots(2, 1, figsize=(15,2), dpi=300, sharex='all')

ax = axes[0]
bs = 1000
nbins = (end-start) // bs

x = np.arange(nbins)
ax.plot(x, bintrack(simi_exp, pos[:-1], start, end, bs), linewidth=0.5, color='C1', alpha=0.8)
ax.plot(x, bintrack(simi, pos[:-1], start, end, bs), linewidth=0.5, color='C0', alpha=0.8)
ax.set_xlim([0, nbins])
# ax.fill_between(x, tmp, 0, where=tmp >= 0, facecolor='C0', interpolate=True)
ax.set_ylim([0.5, 1.0])

sels = (simi - simi_exp > 0.08)
print(sels.sum())
ax.scatter(pos[:-1][sels]//bs, np.ones(sels.sum()), c='r')

ax = axes[1]
# ratio[data.getnnz(axis=0)>500] = 0
tmp = bintrack(ratio, pos, start, end, bs)
# tmp = tmp[:(nbins*bs)].reshape((nbins, bs)).mean(axis=1)
ax.plot(x, tmp, linewidth=0.5)
ax.set_xlim([0, nbins])
ax.set_xticks(np.arange(0, nbins+1, 2e5//bs))
ax.set_xticklabels([f'{xx//100000}M' for xx in np.arange(start, end+1, 2e5)])
ax.fill_between(x, tmp, 0, where=tmp >= 0, facecolor='C0', interpolate=True)
ax.set_ylim([0.0, 1.0])
ax.scatter(pos[:-1][sels]//bs, np.ones(sels.sum()), c='r')


39803


In [32]:
sels.shape

(99566,)